# Week 4, Day 3 — `create_agent`, the Agent Layer
### Local Models Edition — by Abhishek

Yesterday you built a tool loop by hand in LangGraph. Today you get all of
that from a single function call — still on a free local model.

`create_agent` is Layer 3: hand it a model, some tools, and a prompt, and it
builds the agent loop for you. What it builds *is* a LangGraph graph — the
kind you assembled by hand yesterday.


## 0. Setup

In [ ]:
import importlib.util
IN_COLAB = importlib.util.find_spec("google.colab") is not None
BACKEND = "huggingface" if IN_COLAB else "ollama"
print(f"Backend: {BACKEND}")


In [ ]:
if BACKEND == "huggingface":
    %pip install -q langchain langgraph langchain-huggingface transformers torch accelerate
else:
    %pip install -q langchain langgraph langchain-ollama ollama langchain-mcp-adapters


In [ ]:
from IPython.display import Image, display
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_tool_call
from langchain_core.tools import tool
from langgraph.checkpoint.memory import MemorySaver

if BACKEND == "huggingface":
    from transformers import pipeline
    from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace
    pipe = pipeline("text-generation", model="Qwen/Qwen2.5-1.5B-Instruct", max_new_tokens=250)
    llm = ChatHuggingFace(llm=HuggingFacePipeline(pipeline=pipe))
else:
    from langchain_ollama import ChatOllama
    llm = ChatOllama(model="llama3.2:3b", temperature=0.3)

print("llm ready")


## Part 1: the simplest agent

An agent in its plainest form is a model with a prompt. `create_agent` accepts
a model **object** (not just a `provider:model` string) — that's what makes it
easy to point at a local model here.


In [ ]:
agent = create_agent(
    model=llm,
    system_prompt="You are a helpful assistant who answers concisely.",
)

result = agent.invoke({"messages": [{"role": "user", "content": "What is the Model Context Protocol, in two sentences?"}]})
print(result["messages"][-1].content)


### And its async twin: `ainvoke`

In [ ]:
result = await agent.ainvoke({"messages": [{"role": "user", "content": "In one sentence: why does async code suit agents so well?"}]})
print(result["messages"][-1].content)


## Part 2: tools

Give the agent tools. Same `@tool` functions as Day 1; the agent runs the
whole tool loop for us.


In [ ]:
@tool
def get_weather(city: str) -> str:
    """Return today's weather for a city."""
    pretend = {"London": "rainy, 14 degrees", "Rome": "sunny, 27 degrees"}
    return pretend.get(city, "clear, 20 degrees")

@tool
def get_population(city: str) -> str:
    """Return the population of a city."""
    pretend = {"London": "8.9 million", "Rome": "2.8 million"}
    return pretend.get(city, "unknown")

agent = create_agent(
    model=llm,
    tools=[get_weather, get_population],
    system_prompt="You are a travel assistant. Use your tools to answer questions about cities.",
)

result = agent.invoke({"messages": [{"role": "user", "content": "What is the weather and population of Rome?"}]})
print(result["messages"][-1].content)


## The reveal: it is a LangGraph graph

Because `create_agent` returns a compiled LangGraph graph, render it exactly
the way you did yesterday.


In [ ]:
display(Image(agent.get_graph().draw_mermaid_png()))


## Part 3: memory

In [ ]:
memory_agent = create_agent(
    model=llm,
    tools=[get_weather],
    checkpointer=MemorySaver(),
    system_prompt="You are a travel assistant. Use your tools to answer questions about cities.",
)

config = {"configurable": {"thread_id": "trip-planning"}}
memory_agent.invoke({"messages": [{"role": "user", "content": "I am planning a trip to London."}]}, config=config)
result = memory_agent.invoke({"messages": [{"role": "user", "content": "What is the weather like where I am going on my trip?"}]}, config=config)
print(result["messages"][-1].content)


## Part 4: structured output

Pass a Pydantic model as `response_format`. As noted Day 1, smaller local
models are less reliable here than frontier models — a real trade-off worth
discussing with students, not a bug to paper over.


In [ ]:
class CityReport(BaseModel):
    city: str = Field(description="The city name")
    weather: str = Field(description="A short weather description")
    population: str = Field(description="The population")

report_agent = create_agent(
    model=llm,
    tools=[get_weather, get_population],
    response_format=CityReport,
)

result = report_agent.invoke({"messages": [{"role": "user", "content": "Give me a report on London."}]})
report = result["structured_response"]
print(report)
print("Just the weather:", report.weather)


## Part 5: middleware

Middleware runs your own code at fixed points in the loop.


In [ ]:
@wrap_tool_call
def log_tool_calls(request, handler):
    call = request.tool_call
    print(f"  [middleware] calling {call['name']} with {call['args']}")
    return handler(request)

watched_agent = create_agent(
    model=llm,
    tools=[get_weather, get_population],
    system_prompt="You are a travel assistant. Use your tools.",
    middleware=[log_tool_calls],
)

result = watched_agent.invoke({"messages": [{"role": "user", "content": "Weather and population of London and Rome?"}]})
print("\nFinal answer:", result["messages"][-1].content)


LangChain also ships ready-made middleware: `SummarizationMiddleware` to stay
within context, `PIIMiddleware` to redact sensitive data, retry / call-limit
middleware, and `HumanInTheLoopMiddleware` to pause for human approval — none
of these are provider-specific, so they all work here on local models too.


## Before Part 6: Node and Playwright

The last part uses tools that live in a separate MCP server, running on Node.
This whole part is **already free and local** — no API cost at all, no
adaptation needed from the original course.

- **Windows**, PowerShell: `winget install OpenJS.NodeJS.LTS`
- **Mac**, Terminal: `brew install node`
- **Linux**: see your distro's Node install guide (v22+)

After installing, restart your kernel/notebook fully before continuing.


In [ ]:
import subprocess
print(subprocess.run(["node", "--version"], capture_output=True, text=True).stdout)
print(subprocess.run(["npx", "--version"], capture_output=True, text=True).stdout)


`npx` fetches Playwright on demand and drives the copy of Chrome already on
your machine — nothing else to install.


In [ ]:
import subprocess
subprocess.run(["npx", "-y", "playwright@latest", "screenshot", "--channel=chrome",
                 "https://news.ycombinator.com", "playwright_check.png"])
from IPython.display import Image, display
display(Image("playwright_check.png"))


## Part 6: an MCP server, and a real browser

The Model Context Protocol is a standard way for agents to use tools that live
in a separate server. `langchain-mcp-adapters` loads those tools; from the
agent's point of view they're just tools like any other. Connecting to
Microsoft's Playwright MCP server is the **same code regardless of which
model powers the agent** — this is the payoff of the abstraction layers.


In [ ]:
import sys
if sys.platform == "win32":
    import subprocess
    from functools import partial
    import langchain_mcp_adapters.sessions as mcp_sessions
    mcp_sessions.stdio_client = partial(mcp_sessions.stdio_client, errlog=subprocess.DEVNULL)
    print("Applied the Windows adjustment")
else:
    print("Not Windows, nothing to do here")


In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient({
    "playwright": {
        "transport": "stdio",
        "command": "npx",
        "args": ["-y", "@playwright/mcp@latest", "--isolated"],
    }
})

browser_tools = await client.get_tools()
print(f"Loaded {len(browser_tools)} browser tools:")
for t in browser_tools:
    print(" -", t.name)


In [ ]:
browser_agent = create_agent(
    model=llm,
    tools=browser_tools,
    system_prompt="You are a web research assistant. Use the browser tools to complete the task, then report clearly.",
)

result = await browser_agent.ainvoke({"messages": [{"role": "user",
    "content": "Go to https://news.ycombinator.com and tell me the titles of the top three stories on the front page."}]})
print(result["messages"][-1].content)


A note for local models: browser-driving agents need to chain several tool
calls in sequence and reason about page content — this is one of the harder
tasks for a 3B model. If it struggles, try `qwen2.5:7b` if your hardware
allows, or fall back to a hosted model for this specific lab.

## Recap, and where we are heading
You built an agent in a single line, saw it's a LangGraph graph, gave it
tools, memory, structured output, middleware, and a real browser through
MCP — on a free local model throughout.

Tomorrow, Layer 4: Deep Agents.

## Exercise
Give the browser agent a checkpointer and a `thread_id`, hold a short back
and forth, and confirm it remembers. For a bigger challenge, write middleware
that refuses navigation to a site on a block list, and prove it works.
